# Dreamer4 Checkpoint Visualization

Loads a `DynamicsCheckpointBundle` and inspects its structure using [Treescope](https://github.com/google-deepmind/treescope).

In [1]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"  # keep GPUs free for training

import sys
sys.path.insert(0, '..')

import jax
import jax.numpy as jnp
import jax.tree_util as jtu
import numpy as np
from flax import nnx
import treescope

from dreamer.checkpointing import DynamicsCheckpointBundle
from dreamer.parallel import build_parallel

In [2]:
# Register treescope as the default pretty-printer and enable array visualization
treescope.basic_interactive_setup()
treescope.active_autovisualizer.set_globally(
    treescope.ArrayAutovisualizer(maximum_size=4096)
)

## Load checkpoint

`build_parallel("data")` creates a JAX mesh (required by Flax's sharding machinery even on a single device).
Passing `model_names={"dynamics_ema", "tokenizer"}` skips restoring the optimizer state, which is large and not needed here.

In [4]:
DYNAMICS_CKPT = "/shm/dynamics-depth24-0.9b1-0.95b2-20kwarmup/checkpoints"
# Other available: /shm/dynamics-depth24-0.99b1-0.999b2-20kwarmup/checkpoints
# When logs/dynamics-depth-30-spikenomore-20kwarmup is trained, point here instead:
# DYNAMICS_CKPT = "../logs/dynamics-depth-30-spikenomore-20kwarmup/checkpoints"

mesh, _, mesh_rules = build_parallel("data")

with mesh:
    bundle = DynamicsCheckpointBundle.from_pretrained(
        DYNAMICS_CKPT,
        mesh_rules=mesh_rules,
        model_names={"dynamics_ema", "tokenizer"},
    )
print(f"Loaded dynamics: depth={bundle.dynamics_ema.cfg.depth}, d_model={bundle.dynamics_ema.cfg.d_model}, k_max={bundle.dynamics_ema.cfg.k_max}")
print(f"Loaded tokenizer: d_bottleneck={bundle.tokenizer.cfg.encoder.d_bottleneck}, n_latents={bundle.tokenizer.cfg.encoder.n_latents}")

[parallel] data: {'data': 1, 'model': 1}
Loaded dynamics: depth=24, d_model=1536, k_max=256
Loaded tokenizer: d_bottleneck=16, n_latents=512


## Configs

In [5]:
treescope.show("Dynamics EMA config:", bundle.dynamics_ema.cfg)

Dynamics EMA config: DynamicsModelConfig(
  d_bottleneck=16,
  depth=24,
  d_model=1536,
  n_heads=24,
  n_kv_heads=3,
  packing_factor=2,
  n_register=32,
  qk_norm_type='qknorm',
  rope_theta=10000.0,
  time_every=4,
  time_layer_offset=0,
  mlp_ratio=4,
  dropout_rate=0.0,
  use_residual_lambdas=False,
  use_bias=False,
  use_rmsnorm_scale=True,
  use_seq_parallel=False,
  dtype='bfloat16',
  param_dtype='float32',
  use_depth_scaled_init=False,
  use_embed_ln=False,
  k_max=256,
  context_length=64,
  num_binary_actions=27,
  categorical_action_dim=121,
  continuous_action_dim=0,
  latent_mean=[-0.01275550201535225, -0.04425295069813728, 0.08248031884431839, 0.042714960873126984, 0.008957703597843647, -0.0018820130499079823, -0.012893370352685452, -0.08244539052248001, 0.011176219210028648, -0.09440866857767105, -0.05825792998075485, -0.0497550331056118, -0.0025538108311593533, 0.04231492802500725, -0.06914764642715454, 0.07559845596551895],
  latent_std=[0.0848616436123848, 0.09066474437713623, 0.10068009793758392, 0.09407731145620346, 0.1937398761510849, 0.08929944783449173, 0.0907244011759758, 0.10503409057855606, 0.1614910215139389, 0.1081596165895462, 0.15006859600543976, 0.12637652456760406, 0.0923914909362793, 0.10154268890619278, 0.10797301679849625, 0.10930383950471878],
)

In [6]:
treescope.show("Tokenizer config:", bundle.tokenizer.cfg)

Tokenizer config: TokenizerModelConfig(
  encoder=EncoderModelConfig(n_latents=512, d_bottleneck=16, depth=12, d_model=1536, n_heads=24, n_kv_heads=3, patch_size=16, dropout_rate=0.0, qk_norm_type='qknorm', rope_theta=10000.0, time_every=4, time_layer_offset=3, mae_p_min=0.0, mae_p_max=0.9, use_residual_lambdas=False, use_bias=False, use_rmsnorm_scale=True, use_seq_parallel=False, dtype='bfloat16', param_dtype='float32', context_length=16, dataset_mean=[0.2241, 0.2348, 0.2086], dataset_std=[0.1809, 0.1874, 0.2282]),
  decoder=DecoderModelConfig(n_latents=512, d_bottleneck=16, depth=8, d_model=1024, n_heads=16, n_kv_heads=2, patch_size=16, d_patch=768, dropout_rate=0.0, qk_norm_type='qknorm', rope_theta=10000.0, time_every=4, time_layer_offset=3, use_residual_lambdas=False, use_bias=False, use_rmsnorm_scale=True, use_seq_parallel=False, dtype='bfloat16', param_dtype='float32', context_length=16, H=368, W=640, dataset_mean=[0.2241, 0.2348, 0.2086], dataset_std=[0.1809, 0.1874, 0.2282]),
)

## Model structure

Click `▶` to expand submodules. Each `nnx.Param` shows its shape and dtype.

In [7]:
treescope.display(bundle.dynamics_ema)

## Parameter pytree with inline array visualization

`nnx.state()` extracts a pure pytree of `nnx.VariableState` leaves — no module logic, just the weight tensors.
With `ArrayAutovisualizer` enabled globally, each array is rendered as a color-coded heatmap inline.

In [8]:
state = nnx.state(bundle.dynamics_ema)
treescope.display(state)

## Individual layer weight visualization

The transformer layers alternate between `SpaceSelfAttention` and `TimeSelfAttention`.
Both contain a `GroupedQueryAttention` with `to_q`, `to_kv`, `to_out` projections.

In [ ]:
# Layer 0 Q-projection kernel: shape (d_model, d_model)
layer0 = bundle.dynamics_ema.transformer.layers[0]
q_kernel = layer0.attn.attn.to_q.kernel.value
print(f"Layer type: {'time' if layer0.is_time_layer else 'space'}, Q kernel shape: {q_kernel.shape}")
treescope.render_array(q_kernel, around_zero=True)

Layer type: time, Q kernel shape: (1536, 1536)


In [ ]:
# spatial_proj kernel: shape (d_bottleneck * packing_factor, d_model)
spatial_kernel = bundle.dynamics_ema.spatial_proj.kernel.value
print(f"spatial_proj kernel shape: {spatial_kernel.shape}")
treescope.render_array(spatial_kernel, around_zero=True)

In [ ]:
# flow_x_head (output head, zero-initialized): shape (d_model, d_bottleneck * packing_factor)
head_kernel = bundle.dynamics_ema.flow_x_head.kernel.value
print(f"flow_x_head kernel shape: {head_kernel.shape}")
treescope.render_array(head_kernel, around_zero=True)

## Layer-by-layer weight norms

Plot the Frobenius norm of each transformer layer's Q, K, V projections to spot initialization issues or training artifacts.

In [ ]:
rows = []
for i, layer in enumerate(bundle.dynamics_ema.transformer.layers):
    gqa = layer.attn.attn
    q_norm = float(jnp.linalg.norm(gqa.to_q.kernel.value))
    kv_norm = float(jnp.linalg.norm(gqa.to_kv.kernel.value))
    out_norm = float(jnp.linalg.norm(gqa.to_out.kernel.value))
    rows.append({"layer": i, "type": "time" if layer.is_time_layer else "space",
                 "q_norm": q_norm, "kv_norm": kv_norm, "out_norm": out_norm})

treescope.show("Layer weight norms:", rows)

## Full parameter statistics

In [18]:
leaves_with_paths = jtu.tree_leaves_with_path(nnx.state(bundle.dynamics_ema))

rows = []
total_params = 0
for path, arr in leaves_with_paths:
    n = arr.size
    total_params += n
    name = ".".join(str(p.key) if hasattr(p, "key") else str(p.idx) if hasattr(p, "idx") else str(p) for p in path)
    rows.append({
        "name": name,
        "shape": arr.shape,
        "dtype": str(arr.dtype),
        "n_params": n,
        "mean": float(jnp.mean(arr.astype(jnp.float32))),
        "std": float(jnp.std(arr.astype(jnp.float32))),
    })

print(f"Total dynamics_ema parameters: {total_params:,}")
treescope.show("Parameter table:", rows)

Total dynamics_ema parameters: 808,950,528


Parameter table: [
  {'name': 'action_encoder.base_action_emb..value', 'shape': (1536,), 'dtype': 'float32', 'n_params': 1536, 'mean': 0.00011965907469857484, 'std': 0.022017262876033783},
  {'name': 'action_encoder.binary_embeds_list.0.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.01815703883767128, 'std': 1.0048480033874512},
  {'name': 'action_encoder.binary_embeds_list.1.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.008282081224024296, 'std': 1.0088599920272827},
  {'name': 'action_encoder.binary_embeds_list.2.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.007654819171875715, 'std': 1.008484125137329},
  {'name': 'action_encoder.binary_embeds_list.3.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.017450951039791107, 'std': 0.9920745491981506},
  {'name': 'action_encoder.binary_embeds_list.4.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.014496751129627228, 'std': 1.0086848735809326},
  {'name': 'action_encoder.binary_embeds_list.5.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.0051635876297950745, 'std': 1.016754388809204},
  {'name': 'action_encoder.binary_embeds_list.6.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.017327915877103806, 'std': 1.0049861669540405},
  {'name': 'action_encoder.binary_embeds_list.7.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.02218707650899887, 'std': 1.0115686655044556},
  {'name': 'action_encoder.binary_embeds_list.8.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.00789470225572586, 'std': 1.0100914239883423},
  {'name': 'action_encoder.binary_embeds_list.9.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.0010400516912341118, 'std': 0.9799675941467285},
  {'name': 'action_encoder.binary_embeds_list.10.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.0023130232002586126, 'std': 1.0027878284454346},
  {'name': 'action_encoder.binary_embeds_list.11.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.03404031693935394, 'std': 1.0008801221847534},
  {'name': 'action_encoder.binary_embeds_list.12.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.010887941345572472, 'std': 1.0023612976074219},
  {'name': 'action_encoder.binary_embeds_list.13.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.009615044109523296, 'std': 0.9958114624023438},
  {'name': 'action_encoder.binary_embeds_list.14.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.02020011842250824, 'std': 1.0237152576446533},
  {'name': 'action_encoder.binary_embeds_list.15.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.010797533206641674, 'std': 0.9916714429855347},
  {'name': 'action_encoder.binary_embeds_list.16.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.02501567080616951, 'std': 0.9906640648841858},
  {'name': 'action_encoder.binary_embeds_list.17.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.004406120628118515, 'std': 1.0055266618728638},
  {'name': 'action_encoder.binary_embeds_list.18.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': 0.01623750664293766, 'std': 1.0179259777069092},
  {'name': 'action_encoder.binary_embeds_list.19.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0.014117967337369919, 'std': 0.9973128437995911},
  {'name': 'action_encoder.binary_embeds_list.20.embedding..value', 'shape': (2, 1536), 'dtype': 'float32', 'n_params': 3072, 'mean': -0